<a href="https://colab.research.google.com/github/okayren/Palepal-Skripsi/blob/master/model_training/Palepal_dataset_kuku.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Mon Jul 13 09:23:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install ultralytics imagehash -q

In [ ]:
!unzip -q -o Kuku-Anemia.v3i.folder.zip -d dataset_kuku
print("Ekstrak Selesai 100%! Folder dataset_kuku siap digunakan.")

Ekstrak Selesai 100%! Folder dataset_kuku siap digunakan.


In [ ]:
!find /content/dataset_kuku -maxdepth 2 -type d

/content/dataset_kuku
/content/dataset_kuku/train
/content/dataset_kuku/train/Non_Anemia
/content/dataset_kuku/train/Anemia
/content/dataset_kuku/valid
/content/dataset_kuku/valid/Non_Anemia
/content/dataset_kuku/valid/Anemia
/content/dataset_kuku/test
/content/dataset_kuku/test/Non_Anemia
/content/dataset_kuku/test/Anemia


In [ ]:
import os

dataset_root = "/content/dataset_kuku"

if os.path.exists(os.path.join(dataset_root, "valid")) and not os.path.exists(os.path.join(dataset_root, "val")):
    os.rename(os.path.join(dataset_root, "valid"), os.path.join(dataset_root, "val"))
    print("Folder 'valid' berhasil di-rename jadi 'val'")
else:
    print("Tidak perlu rename, atau folder 'val' sudah ada / 'valid' tidak ditemukan.")

!find /content/dataset_kuku -maxdepth 2 -type d

Folder 'valid' berhasil di-rename jadi 'val'
/content/dataset_kuku
/content/dataset_kuku/train
/content/dataset_kuku/train/Non_Anemia
/content/dataset_kuku/train/Anemia
/content/dataset_kuku/test
/content/dataset_kuku/test/Non_Anemia
/content/dataset_kuku/test/Anemia
/content/dataset_kuku/val
/content/dataset_kuku/val/Non_Anemia
/content/dataset_kuku/val/Anemia


In [ ]:
print(f"\n{'='*60}")
print("DISTRIBUSI KELAS KUKU PER SPLIT")
print(f"{'='*60}")

for split in ["train", "val", "test"]:
    split_dir = os.path.join(dataset_root, split)
    if not os.path.exists(split_dir):
        continue
    print(f"\n[{split}]")
    for cls in os.listdir(split_dir):
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        n = len([f for f in os.listdir(cls_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
        print(f"  {cls}: {n} gambar")


DISTRIBUSI KELAS KUKU PER SPLIT

[train]
  Non_Anemia: 477 gambar
  Anemia: 642 gambar

[val]
  Non_Anemia: 44 gambar
  Anemia: 63 gambar

[test]
  Non_Anemia: 19 gambar
  Anemia: 34 gambar


In [ ]:
import imagehash
from PIL import Image

HASH_THRESHOLD = 5

def compute_hashes(folder):
    hashes = {}
    if not os.path.exists(folder):
        return hashes
    for f in os.listdir(folder):
        if not f.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        try:
            hashes[f] = imagehash.phash(Image.open(os.path.join(folder, f)))
        except Exception as e:
            print(f"Gagal baca {f}: {e}")
    return hashes

print(f"\n\n{'='*60}")
print("CEK LEAKAGE VISUAL ANTAR SPLIT (KUKU)")
print(f"{'='*60}")

split_names = ["train", "val", "test"]
class_names = os.listdir(os.path.join(dataset_root, "train"))

for cls in class_names:
    print(f"\n--- Kelas: {cls} ---")
    split_hashes = {}
    for split in split_names:
        folder = os.path.join(dataset_root, split, cls)
        split_hashes[split] = compute_hashes(folder)
        print(f"  {split}: {len(split_hashes[split])} gambar")

    for i in range(len(split_names)):
        for j in range(i + 1, len(split_names)):
            s1, s2 = split_names[i], split_names[j]
            h1, h2 = split_hashes[s1], split_hashes[s2]
            if not h1 or not h2:
                continue
            found = []
            for f1, hash1 in h1.items():
                for f2, hash2 in h2.items():
                    dist = hash1 - hash2
                    if dist <= HASH_THRESHOLD:
                        found.append((f1, f2, dist))
            print(f"  [{s1} <-> {s2}] gambar mirip ditemukan: {len(found)}")
            if found:
                print(f"    Contoh (max 5):")
                for f1, f2, dist in found[:5]:
                    print(f"      {s1}/{f1}  <->  {s2}/{f2}   (distance={dist})")



CEK LEAKAGE VISUAL ANTAR SPLIT (KUKU)

--- Kelas: Non_Anemia ---
  train: 477 gambar
  val: 44 gambar
  test: 19 gambar
  [train <-> val] gambar mirip ditemukan: 2
    Contoh (max 5):
      train/Non-Anrmic-FN-157_png.rf.56963cbd1adc3c0b2f4b7f7335ff493d.jpg  <->  val/Non-anemic-Fin-005_png.rf.0b1c3824b4282b81021df2b0386827ad.jpg   (distance=4)
      train/Non-Anrmic-FN-013_png.rf.2dfae29471125f4a688cbbb5487a9949.jpg  <->  val/Non-Anrmic-FN-014_png.rf.ccc279099d2dff6bc5b47ca021c57076.jpg   (distance=4)
  [train <-> test] gambar mirip ditemukan: 0
  [val <-> test] gambar mirip ditemukan: 0

--- Kelas: Anemia ---
  train: 642 gambar
  val: 63 gambar
  test: 34 gambar
  [train <-> val] gambar mirip ditemukan: 4
    Contoh (max 5):
      train/Anemic-FN-254_png.rf.b1de114d1641463f2ca3171e6c01972b.jpg  <->  val/Anemic-FN-109_png.rf.c41afe631e2eff094cf9b29d3ef17649.jpg   (distance=4)
      train/Anmeic-fn-0021-1-5_png.rf.b108b7acfe99440eab7a6ac9b645f9c3.jpg  <->  val/Anmeic-fn-0023-2_png.rf

In [ ]:
print(f"\n\n{'='*60}")
print("CONTOH NAMA FILE (untuk cek pola nama orang, kalau ada)")
print(f"{'='*60}")
for cls in class_names:
    cls_dir = os.path.join(dataset_root, "train", cls)
    files = os.listdir(cls_dir)[:5]
    print(f"\n[{cls}]")
    for f in files:
        print(f"  {f}")



CONTOH NAMA FILE (untuk cek pola nama orang, kalau ada)

[Non_Anemia]
  Non-anemic-Fin-009_png.rf.c8f9c6f4bff4ae3c674c7f36dc3f44d4.jpg
  Non-anemic-Fin-002_png.rf.b30510a9058628e0b7f43469f3bbc7bf.jpg
  Non-Anrmic-FN-140_png.rf.183be20f800a6fc50c0555fddb4ec12b.jpg
  Non-Anrmic-FN-056_png.rf.af70e812b19431961ce6a53da37f8c00.jpg
  Non-Anrmic-FN-017_png.rf.528140f4b5b9afce09e7237dfbb97a66.jpg

[Anemia]
  Anemic-Fin-023_png.rf.caa5029b0fc2fc954e8dbd62b363aa75.jpg
  Anemic-Fin-019_png.rf.c30f44b465b4d4fef1d31b2ddcde8597.jpg
  Anemic-FN-108_png.rf.03d6bd414eb9d71a4338983f79d239ca.jpg
  Anemic-FN-034_png.rf.18c61ca8dd6e0793add113afa784a9f6.jpg
  Anemic-FN-254_png.rf.47e3a8fbbcbc2117b348a4d780701b38.jpg


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n-cls.pt')

results = model.train(
    data="/content/dataset_kuku",
    epochs=50,
    imgsz=512,
    plots=True,
    degrees=15.0,
    fliplr=0.5,
    hsv_s=0.2,
    hsv_v=0.2,
    scale=0.1
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.93 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_kuku, degrees=15.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, h

In [ ]:
import glob

train_dirs = sorted(glob.glob('/content/runs/classify/train*'), key=os.path.getmtime)
latest_train_dir = train_dirs[-1]
print("Folder training terbaru:", latest_train_dir)

model_kuku = YOLO(f'{latest_train_dir}/weights/best.pt')

print("Urutan kelas model kuku:", model_kuku.names)

model_kuku.export(format='tflite', imgsz=512)
print("Konversi selesai!")

Folder training terbaru: /content/runs/classify/train
Urutan kelas model kuku: {0: 'Anemia', 1: 'Non_Anemia'}
WARNING ⚠️ format='tflite' is deprecated as of 8.4.83 and has been replaced by the unified Google LiteRT format. Exporting format='litert' instead. See https://docs.ultralytics.com/integrations/litert/
Ultralytics 8.4.93 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs

PyTorch: starting from '/content/runs/classify/train/weights/best.pt' with input shape (1, 3, 512, 512) BCHW and output shape(s) (1, 2) (2.8 MB)
requirements: Ultralytics requirements ['litert-torch>=0.9.0', 'ai-edge-litert>=2.1.4'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 85 packages in 529ms
Prepared 15 packages in 2.97s
Un

/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:1745: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.



LiteRT: starting export with litert_torch 0.9.1...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

...

(00:07) [ DONE] LiteRT-Torch Convert > Run LiteRT Converter Passes (+00:00)

(00:07) [ DONE] LiteRT-Torch Convert (+00:07)

(00:00) [START] Write Model to /content/runs/classify/train/weights/best.tflite

(00:00) [ DONE] Write Model to /content/runs/classify/train/weights/best.tflite (+00:00)

LiteRT: export success ✅ 13.5s, saved as '/content/runs/classify/train/weights/best.tflite' (5.5 MB)

Export complete (13.7s)
Results saved to /content/runs/classify/train/weights/best.tflite
Predict:         yolo predict task=classify model=/content/runs/classify/train/weights/best.tflite imgsz=512 
Validate:        yolo val task=classify model=/content/runs/classify/train/weights/best.tflite imgsz=512 data=/content/dataset_kuku  
Visualize:       https://netron.app
Konversi selesai!


In [ ]:
!find {latest_train_dir} -name "*.tflite"

import shutil
shutil.make_archive('Model_Palepal_Kuku_CLS', 'zip', latest_train_dir)
print("File 'Model_Palepal_Kuku_CLS.zip' siap didownload.")

/content/runs/classify/train/weights/best.tflite
File 'Model_Palepal_Kuku_CLS.zip' siap didownload.


In [ ]:
from ultralytics import YOLO

# Sesuaikan path ini dengan lokasi weights model kuku kamu
# (biasanya di /content/runs/classify/train*/weights/best.pt)
model = YOLO("/content/runs/classify/train/weights/best.pt")

metrics = model.val(data="/content/dataset_kuku", split="test")
print("Top-1 accuracy TEST (kuku):", metrics.top1)

Ultralytics 8.4.93 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: /content/dataset_kuku/train... found 1119 images in 2 classes ✅ 
val: /content/dataset_kuku/val... found 107 images in 2 classes ✅ 
test: /content/dataset_kuku/test... found 53 images in 2 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 215.8±102.1 MB/s, size: 6.8 KB)
test: Scanning /content/dataset_kuku/test... 53 images, 0 corrupt: 100% ━━━━━━━━━━━━ 53/53 4.9Kit/s 0.0s
test: New cache created: /content/dataset_kuku/test.cache


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 4/4 2.4it/s 1.7s
                   all      0.679          1
Speed: 4.5ms preprocess, 14.6ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/val
Top-1 accuracy TEST (kuku): 0.6792452931404114
